# Using LLMs and Knowledge graphs to search for PFAS Alternatives

## Project with Saint Gobain

#### Yu-Chuan (Michael) Hsu, Isabella Stewart, Tarjei Hage, Wei Lu, and Markus J. Buehler, MIT, 2025 
mkychsu@MIT.EDU, istewart@MIT.EDU, tphage@MIT.EDU, wl7@MIT.EDU, mbuehler@MIT.EDU
#### LAMM, Massachusetts Institute of Technology


# Allows for distributed or parallel processing of a dataset

In [ ]:
import sys, os

try:
    thread_i = int(sys.argv[1]) #which thread number this process is (e.g., in multi-threaded runs)
    total_threads = int(sys.argv[2]) #how many total threads are running

except: 
    thread_i = 0 #If no arguments are provided (e.g. during a notebook run), it defaults to a single-threaded run
    total_threads = 1
    merge_every = 100

In [ ]:

from dotenv import load_dotenv
load_dotenv()

config_list = [
    {
        "model": os.getenv("URL"),     
        "api_key": os.getenv("KEY"),
        "max_tokens": 20000
    },
]

In [ ]:
config_list

# Client Initiation with Together

In [ ]:
from langchain_openai import ChatOpenAI

client = ChatOpenAI(model=config_list[0]["model"],
                    temperature=0,
                    max_tokens=config_list[0]["max_tokens"],
                    api_key=config_list[0]["api_key"]
                    )

In [ ]:
#import os
#from GraphReasoning import *

import sys
sys.path.insert(0, '/orcd/home/002/istewart/orcd/pool/hypergraph/GraphReasoning_SG') #change functions here. 

In [ ]:
verbatim=False

In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name.lower() == "sg" and cwd.parent.name.lower() == "notebooks":
    workspace_root = cwd.parent.parent
else:
    workspace_root = cwd

doc_data_dir = str((workspace_root / 'Data').resolve())  # place where you keep your markdown files
data_dir='./GRAPHDATA_paper'    
data_dir_output='./GRAPHDATA_OUTPUT_paper'

max_tokens = config_list[0]['max_tokens']

embedding_file='hypergraph_embeedings.pkl' #what your embedding file will be 


# Embedding the graph with Nomic

In [ ]:
if total_threads == 1: ##merging mode: This block only runs if you're not distributing work across multiple threads.

    from transformers import AutoModelForCausalLM, AutoTokenizer
    from sentence_transformers import SentenceTransformer
    embedding_tokenizer =''
    embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

    from GraphReasoning import load_embeddings, save_embeddings, generate_hypernode_embeddings
    
    import hypernetx as hnx

    import torch
    generate_new_embeddings=True

    if os.path.exists(f'{data_dir}/{embedding_file}'):
        print('Found existing embedding file')
        generate_new_embeddings=False
    
    # generate_new_embeddings=True

    with torch.no_grad():
        if generate_new_embeddings:
            H = hnx.Hypergraph({})
            # Extract node IDs (will be empty here)
            nodes = list(H.nodes)
            # Generate embeddings for (new) nodes
            node_embeddings = generate_hypernode_embeddings(
                nodes,
                embedding_tokenizer,
                embedding_model,
            )
            # Save them
            save_embeddings(node_embeddings, f'{data_dir}/{embedding_file}')
        else:
            # Load previously computed embeddings
            node_embeddings = load_embeddings(f'{data_dir}/{embedding_file}')                 

### Load dataset

In [ ]:
### Load dataset of papers

import pandas as pd
import glob

### Load all markdown files (dataset of papers) --- assumes each subfolder in doc_data_dir is named like a paper ID
#doc_list becomes a list of full file paths to .md files

doc_list=[]
with os.scandir(f'{doc_data_dir}') as folders:
    for folder in folders:
        doc_list.append(f'{doc_data_dir}/{folder.name}/{folder.name}.md')

doc_list=sorted(doc_list)

### Set up LLM client:

In [ ]:
import instructor
from typing import List
from PIL import Image
import base64
from pydantic import BaseModel
from instructor import patch
from pathlib import Path
import os

from GraphReasoning.prompt_config import get_prompt


class Event(BaseModel):
    source: List[str]
    target: List[str]
    relation: str


class HypergraphJSON(BaseModel):
    events: List[Event]


response_model = HypergraphJSON
system_prompt = get_prompt("hypergraph", "graphmaker_system")


def _instructor_create():
    return instructor.patch(
        create=client.chat.completions.create,
        mode=instructor.Mode.JSON_SCHEMA,
    )


def generate(
    system_prompt=system_prompt,
    prompt="",
    temperature=0.333,
    max_tokens=config_list[0]["max_tokens"],
    response_model=HypergraphJSON,
):
    if system_prompt is None:
        messages = [{"role": "user", "content": f"{prompt}"}]
    else:
        messages = [
            {"role": "system", "content": f"{system_prompt}"},
            {"role": "user", "content": f"{prompt}"},
        ]

    create = _instructor_create()

    return create(
        messages=messages,
        model=config_list[0]["model"],
        max_tokens=max_tokens,
        temperature=temperature,
        response_model=response_model,
    )


def image_to_base64_data_uri(file_path):
    with open(file_path, "rb") as image_file:
        base64_data = base64.b64encode(image_file.read()).decode("utf-8")
        return f"data:image/png;base64,{base64_data}"


def generate_figure(
    image,
    system_prompt=None,
    prompt="",
    temperature=0,
    max_tokens=config_list[0]["max_tokens"],
):
    pwd = os.getcwd()
    image = image.split(pwd)[-1]
    image = Path(".").glob(f"**/{image}", case_sensitive=False)
    image = list(image)[0]

    image_uri = image_to_base64_data_uri(image)
    messages = [
        {
            "role": "system",
            "content": system_prompt or get_prompt("runtime", "figure_system_prompt"),
        },
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_uri}},
                {"type": "text", "text": prompt or get_prompt("runtime", "figure_user_prompt")},
            ],
        },
    ]

    create = _instructor_create()
    return create(
        messages=messages,
        model=config_list[0]["model"],
        max_tokens=max_tokens,
        temperature=temperature,
        response_model=response_model,
    ).choices[0].message.content

In [ ]:
print(f'running on {thread_i}-th thread in totally {total_threads} threads') #double check threading 

### Checkpoint Cell: Finds where merging left off with current_merged_i. Handles resuming graph merging in single-threaded mode

#### 1. Finds the latest merged graph from previous runs.

#### 2. Checks if it's valid by trying to load it.

#### 3. If the graph or its parts are corrupted, it deletes them and rolls back to the previous merge index.

#### 4. Sets current_merged_i so merging can resume cleanly.

In [ ]:
# Threading / merging config + robust resume indexing
import os, sys, re, glob

try:
    thread_i = int(sys.argv[1])
    total_threads = int(sys.argv[2])
except Exception:
    thread_i = 0
    total_threads = 1

merge_every = 10

INT_PREFIX = re.compile(r'^(\d+)_')
def extract_idx(path: str) -> int:
    name = os.path.basename(path)
    m = INT_PREFIX.match(name)
    return int(m.group(1)) if m else -1

# Directories
cwd = Path.cwd().resolve()
if cwd.name.lower() == "sg" and cwd.parent.name.lower() == "notebooks":
    workspace_root = cwd.parent.parent
else:
    workspace_root = cwd

doc_data_dir = str((workspace_root / 'Data').resolve())
data_dir = './GRAPHDATA_paper'
data_dir_output = './GRAPHDATA_OUTPUT_paper'

if total_threads == 1:  # merging mode
    # 1) find all "*_integrated.pkl" files
    merged_graph_list = sorted(
        glob.glob(f'{data_dir_output}/*_integrated.pkl'),
        reverse=True,
        key=extract_idx
    )

    # 2) determine current merged index; fall back to 0 if none found
    current_merged_i = extract_idx(merged_graph_list[0]) if merged_graph_list else 0

    # 3) find any per-session files for the current index
    last_graph = sorted(
        glob.glob(f'{data_dir}/{current_merged_i}_*.pkl'),
        reverse=True,
        key=extract_idx
    )
    if last_graph:
        graph_pkl = last_graph[0]
    else:
        graph_pkl = None
else:
    current_merged_i = 0
    graph_pkl = None

print('Thread:', thread_i, '/', total_threads)
print('merge_every:', merge_every)
print('current_merged_i:', current_merged_i)
print('graph_pkl:', graph_pkl)
print('doc_data_dir:', doc_data_dir)

### Generate a Knowledge Graph (KG) from each document using an LLM + embedding pipeline, then (optionally) merge it into a global graph in merging mode.

In [ ]:
import os
import sys
import pickle
import hypernetx as hnx
from datetime import datetime
import time
import torch
import shutil
import traceback

# Prefer local workspace package over site-packages
candidate_roots = [os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..', '..'))]
for root in candidate_roots:
    if os.path.exists(os.path.join(root, 'GraphReasoning')) and root not in sys.path:
        sys.path.insert(0, root)
        break

from GraphReasoning.graph_generation import make_hypergraph_from_text
from GraphReasoning.graph_tools import add_new_hypersubgraph_from_text, update_hypernode_embeddings, save_embeddings

# Initialize the "global" graph
try:
    G = hnx.Hypergraph({})
except Exception:
    G = None

with torch.no_grad():
    for i, doc in enumerate(doc_list):
        # only process docs for this thread
        if i % total_threads != thread_i:
            continue
        # skip already-merged docs
        if i < current_merged_i:
            continue

        # extract title/doi and text
        title = os.path.basename(doc).rsplit('.md', 1)[0]
        doi = title
        with open(doc, 'r') as f:
            txt = f.read()

        # define where this doc's subgraph lives
        graph_root = f'{i}_{title[:100]}'
        current_graph = os.path.join(data_dir, f'{graph_root}.pkl')

        # Variable to store current document's sub_dfs
        current_doc_sub_dfs = None
        
        # generate until the subgraph file appears
        while not os.path.exists(current_graph):
            print(f"Generating KG for {i}: {title}")
            try:
                if not isinstance(txt, str):
                    print("Text is not a string:", txt)
                    break
                now = datetime.now()
                # FIX: Capture all return values including sub_dfs
                current_graph, _, sub_dfs_pkl_path, current_doc_sub_dfs = make_hypergraph_from_text(
                    txt, generate, generate_figure, image_list='',
                    graph_root=graph_root,
                    do_distill=False,
                    do_relabel=False,
                    chunk_size=10000, chunk_overlap=0,
                    repeat_refine=0, verbatim=False,
                    data_dir=data_dir,
                )
                print("Time:", datetime.now() - now)
                print(f"[generation] Captured sub_dfs with {len(current_doc_sub_dfs) if current_doc_sub_dfs else 0} chunks")
            except Exception as e:
                print("Error during KG generation:", repr(e))
                time.sleep(60)

        # ── MERGING MODE ──
        if total_threads == 1:

            if i % merge_every == 0:
                do_simplify_graph = True
                size_threshold = 10
            else:
                do_simplify_graph = False
                size_threshold = 0
                
            _hypergraph_pkl = os.path.join(data_dir_output, f'{graph_root}_integrated.pkl')
            # skip if already merged
            if os.path.exists(_hypergraph_pkl):
                with open(_hypergraph_pkl, 'rb') as f:
                    G = pickle.load(f)  
                print(f"[merge] already have integrated at {_hypergraph_pkl}")
                continue            
            
            now = datetime.now()
            print(f"[merge] about to merge paper {i}: will write {_hypergraph_pkl!r}")

            try:
                graph_path = current_graph  # save the path
                with open(graph_path, 'rb') as f:
                    H0 = pickle.load(f)
                current_graph = hnx.Hypergraph(
                    H0.incidence_dict,
                    edge_attr={'DOI': {eid: doi for eid in H0.incidence_dict}}
                )
                print(f"[merge] loaded subgraph with {len(current_graph.edges)} edges")
            except Exception as e:
                print(f"[merge] failed loading/annotating {graph_path!r}: {e!r}")
                continue

            ### FIX: Load cumulative sub_dfs
            updated_path = os.path.join(data_dir_output, "updated_sub_dfs.pkl")
            if os.path.exists(updated_path):
                with open(updated_path, "rb") as f:
                    sub_dfs = pickle.load(f)
                print(f"[merge] Loaded cumulative sub_dfs from {updated_path} with {len(sub_dfs)} existing chunks")
            else:
                sub_dfs = []
                print(f"[merge] Starting fresh sub_dfs list")

            ### FIX: If generation happened (and wasn't skipped), add current doc's chunks
            if current_doc_sub_dfs is None:
                # Generation was skipped (file already existed), load from the saved pkl
                sub_dfs_pkl_path = os.path.join(data_dir, "original_sub_dfs.pkl")
                if os.path.exists(sub_dfs_pkl_path):
                    with open(sub_dfs_pkl_path, "rb") as f:
                        current_doc_sub_dfs = pickle.load(f)
                    print(f"[merge] Loaded current doc's sub_dfs from {sub_dfs_pkl_path}")
            
            ### FIX: Append current document's chunks to cumulative list
            if current_doc_sub_dfs:
                if isinstance(current_doc_sub_dfs, list):
                    sub_dfs.extend(current_doc_sub_dfs)
                    print(f"[merge] Added {len(current_doc_sub_dfs)} chunks from doc {i}. Total: {len(sub_dfs)} chunks")
                else:
                    sub_dfs.append(current_doc_sub_dfs)
                    print(f"[merge] Added 1 chunk from doc {i}. Total: {len(sub_dfs)} chunks")
            else:
                print(f"[merge] WARNING: No sub_dfs found for doc {i}")

            if G is None:
                G = current_graph
                integrated_path = _hypergraph_pkl
                with open(integrated_path, 'wb') as f:
                    pickle.dump(G, f)
            else:
                # perform the merge
                integrated_path, G, _, node_embeddings, sub_dfs = add_new_hypersubgraph_from_text(
                    txt='',
                    node_embeddings=node_embeddings,
                    tokenizer=embedding_tokenizer,
                    model=embedding_model,
                    original_graph=G,
                    data_dir_output=data_dir_output,
                    graph_root=graph_root,
                    do_simplify_graph=do_simplify_graph,
                    do_relabel=False,
                    size_threshold=size_threshold,
                    do_update_node_embeddings=do_simplify_graph,
                    repeat_refine=0,
                    similarity_threshold=0.9,
                    do_Louvain_on_new_graph=False,
                    return_only_giant_component=False,
                    save_common_graph=False,
                    G_to_add=current_graph,
                    graph_pkl_to_add=None, 
                    sub_dfs=sub_dfs,  # Now contains all previous + current doc's chunks
                    verbatim=True,
                )

            print(f"[merge] After merge, sub_dfs contains {len(sub_dfs)} total chunks")

            # ── CONDITIONAL EMBEDDING UPDATE ──
            doc_count = len(doc_list)
            is_last_group = (doc_count < 100) or (i >= (doc_count // 100) * 100)

            if is_last_group:
                try:
                    print(f"[update] Performing embedding update for doc {i} (triggered by small or remainder group)")
                    node_embeddings = update_hypernode_embeddings(node_embeddings, G, embedding_tokenizer, embedding_model)
                except Exception as e:
                    print(f"[update] Failed to update embeddings: {e!r}")

            # ── check consistency ──
            graph_nodes = set(str(n) for n in G.nodes)
            embedding_keys = set(str(k) for k in node_embeddings)
        
            missing = graph_nodes - embedding_keys
            extra = embedding_keys - graph_nodes
        
            if not missing and not extra:
                print(f"[check] Embeddings are aligned with the graph nodes. Count = {len(graph_nodes)}")
            else:
                print(f"[check] Embedding mismatch detected:")
                if missing:
                    print(f"  - Missing embeddings for {len(missing)} nodes: {list(missing)[:5]}")
                if extra:
                    print(f"  - Extra embeddings for {len(extra)} nodes not in graph: {list(extra)[:5]}")
                        
            #rename the very final last saved clean graph 
            is_last_doc = (i == len(doc_list) - 1)
                        
            if is_last_doc:
                final_path = os.path.join(data_dir_output, "final_graph.pkl")
                try:
                    shutil.copyfile(integrated_path, final_path)
                    print(f"[merge] Final graph saved as: {final_path}")
                except Exception as e:
                    print(f"[merge] Failed to rename final graph: {e!r}")
            
            try:
                save_embeddings(node_embeddings, os.path.join(data_dir, embedding_file))
                print("[merge] embeddings saved")
            except Exception as e:
                print(f"[merge] failed to save embeddings: {e!r}")
            
            print("Merge elapsed time:", datetime.now() - now)